# Exercise a read-only MCP library adapter

Use in-memory MCP-shaped fixtures to test a library integration. After the required tasks, the optional local SDK example in the last lesson lets you discover and call the same kind of tool over stdio.

Fill in each starter, then run its check cell. Restart the kernel and run all cells before submitting. Keep your interpretation in the last Markdown cell.


## 1. Plan capability-aware discovery

Check which primitives the server advertised.

Return supported list methods in tools, resources, prompts order. An empty advertised object still counts. Ignore unknown capability keys.


In [ ]:
def discovery_methods(capabilities):
    raise NotImplementedError


In [ ]:
assert discovery_methods({"prompts": {}, "tools": {}}) == ["tools/list", "prompts/list"]
assert discovery_methods({"resources": {"subscribe": True}, "other": {}}) == ["resources/list"]
assert discovery_methods({}) == []
print("OK")


## 2. Resolve a public library resource

Expose a catalogue URI as application-selected data.

Return status ok, URI and text for an exact registry match. For an unknown URI return only status not_found. A stored empty string is a successful read.


In [ ]:
def read_public_resource(uri, resources):
    raise NotImplementedError


In [ ]:
assert read_public_resource("library://a", {"library://a": ""}) == {"status":"ok","uri":"library://a","text":""}
assert read_public_resource("library://missing", {"library://a":"A"}) == {"status":"not_found"}
assert read_public_resource("library://a", {"library://a":"A","library://b":"B"})["text"] == "A"
print("OK")


## 3. Build a tools/call request

Construct and correlate a read-only call.

Return a JSON-RPC 2.0 dictionary with id, method tools/call and params containing name and a shallow copy of arguments. Preserve the provided request ID.


In [ ]:
def tool_request(request_id, name, arguments):
    raise NotImplementedError


In [ ]:
args = {"branch":"north"}
message = tool_request(11,"hours",args)
assert message == {"jsonrpc":"2.0","id":11,"method":"tools/call","params":{"name":"hours","arguments":{"branch":"north"}}}
args["branch"] = "south"
assert message["params"]["arguments"]["branch"] == "north"
assert tool_request(0,"list_public",{})["id"] == 0
print("OK")


## 4. Classify a simulated MCP response

Classify tool, transport and successful outcomes.

Return no_response for None, unexpected_id for mismatch, protocol_error for a protocol error or missing dictionary result, tool_error when result.isError is True, and ok otherwise.


In [ ]:
def response_status(response, expected_id):
    raise NotImplementedError


In [ ]:
assert response_status(None,1) == "no_response"
assert response_status({"id":2,"result":{}},1) == "unexpected_id"
assert response_status({"id":1,"error":{"code":-32602}},1) == "protocol_error"
assert response_status({"id":1,"result":[]},1) == "protocol_error"
assert response_status({"id":1,"result":{"isError":True}},1) == "tool_error"
assert response_status({"id":1,"result":{}},1) == "ok"
assert response_status({"id":2,"error":{"code":-32601}},1) == "unexpected_id", "Check the ID before the error"
assert response_status({"id":1,"result":{"isError":"yes"}},1) == "ok", "Only isError true marks a tool error"
print("OK")


## 5. Replay an MCP tool exchange

Implement replay_mcp(capabilities, send, request_id, name, arguments). The send fixture takes
one request envelope and returns a response or None. If tools are not advertised, return
unsupported without constructing or sending a call. Otherwise build the request and return the
status classified by response_status. Use the original request ID for correlation.


In [ ]:
def replay_mcp(capabilities, send, request_id, name, arguments):
    raise NotImplementedError


In [ ]:
requests = []
def send(message):
    requests.append(message)
    return {"id":message["id"], "result":{"isError":False}}
assert replay_mcp({"resources":{}}, send, 1, "hours", {}) == "unsupported"
assert requests == []
assert replay_mcp({"tools":{}}, send, 2, "hours", {"branch":"north"}) == "ok"
assert requests[0]["params"] == {"name":"hours","arguments":{"branch":"north"}}
assert replay_mcp({"tools":{}}, lambda _: {"id":9,"result":{}}, 2, "hours", {}) == "unexpected_id"
assert replay_mcp({"tools":{}}, lambda _: None, 2, "hours", {}) == "no_response"
print("OK")


## Interpretation

Draw the host, client session and server with the direction of initialize, list and call messages. Where would you enforce permissions if this service also held restricted library records?

**Your answer**
